# 01 - Before / After Preprocessing

Notebook này so sánh nhanh raw train split với cleaned train split.

Trọng tâm:
- giữ ít code hơn
- rút ra nhiều insight hơn
- xem được hiệu quả của preprocessing ngay trên text, labels, và noise

In [5]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 180)

def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Cannot locate project root")

ROOT = find_project_root()
RAW_TRAIN = ROOT / "data/interim/raw_train/train.json"
CLEAN_TRAIN = ROOT / "data/processed/train_clean.json"
RAW_SCAN = ROOT / "experiments/reports/train_scan.json"
CLEAN_SCAN = ROOT / "experiments/reports/train_clean_scan.json"

raw_df = pd.read_json(RAW_TRAIN)
clean_df = pd.read_json(CLEAN_TRAIN)
raw_report = json.loads(RAW_SCAN.read_text(encoding="utf-8"))
clean_report = json.loads(CLEAN_SCAN.read_text(encoding="utf-8"))

print(f"raw rows: {len(raw_df)}")
print(f"clean rows: {len(clean_df)}")
print(f"rows removed: {len(raw_df) - len(clean_df)}")

raw rows: 10728
clean rows: 10696
rows removed: 32


In [6]:
def extract_metrics(report: dict) -> dict:
    checks = report["checks"]
    return {
        "rows": report["metadata"]["row_count"],
        "missing_content": checks["missing_values"]["per_column"]["content"]["missing_count"],
        "short_texts": checks["length"]["shorter_than_min_length"],
        "encoding_rows": checks["encoding"]["rows_with_any_issue"],
        "noise_rows": checks["noise_patterns"]["rows_with_any_noise"],
        "emoji_rows": checks["emoji"]["rows_with_emoji"],
        "duplicates": checks["duplicates"]["normalized_duplicate_texts"],
    }


summary = pd.DataFrame(
    [extract_metrics(raw_report), extract_metrics(clean_report)],
    index=["raw", "clean"],
).T

label_before = raw_df["sentiment_llm"].value_counts().sort_index()
label_after = clean_df["sentiment_llm"].value_counts().sort_index()
label_compare = pd.DataFrame({"raw": label_before, "clean": label_after}).fillna(0).astype(int)

raw_len = raw_df["content"].fillna("").str.len()
clean_len = clean_df["content"].fillna("").str.len()
raw_words = raw_df["content"].fillna("").str.split().map(len)
clean_words = clean_df["content"].fillna("").str.split().map(len)

display(summary)
display(label_compare)

,raw,clean
rows,10728,10696
missing_content,6,0
short_texts,14,0
encoding_rows,212,23
noise_rows,768,649
emoji_rows,338,0
duplicates,11,0


,raw,clean
sentiment_llm,,
0,5622,5608
1,1749,1744
2,3357,3344


In [7]:
issue_df = pd.DataFrame(
    {
        "raw": [
            raw_report["checks"]["encoding"]["rows_with_any_issue"],
            raw_report["checks"]["noise_patterns"]["rows_with_any_noise"],
            raw_report["checks"]["emoji"]["rows_with_emoji"],
            raw_report["checks"]["duplicates"]["normalized_duplicate_texts"],
        ],
        "clean": [
            clean_report["checks"]["encoding"]["rows_with_any_issue"],
            clean_report["checks"]["noise_patterns"]["rows_with_any_noise"],
            clean_report["checks"]["emoji"]["rows_with_emoji"],
            clean_report["checks"]["duplicates"]["normalized_duplicate_texts"],
        ],
    },
    index=["encoding", "noise", "emoji", "duplicates"],
)

changed = clean_df.loc[clean_df["content_raw"].fillna("") != clean_df["content"].fillna("")].copy()
changed["raw_len"] = changed["content_raw"].fillna("").str.len()
changed["clean_len"] = changed["content"].fillna("").str.len()
changed["delta"] = changed["raw_len"] - changed["clean_len"]
changed["word_delta"] = changed["content_raw"].fillna("").str.split().map(len) - changed["content"].fillna("").str.split().map(len)

fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=(
        "Issue counts before vs after",
        "Word count distribution",
        "Label distribution",
        "Length delta distribution",
    ),
)

for col, split, color in [(1, "raw", "#7aa6ff"), (1, "clean", "#2d6cdf")]:
    fig.add_trace(
        go.Bar(name=split, x=issue_df.index, y=issue_df[split], marker_color=color, showlegend=(col == 1)),
        row=1,
        col=1,
    )

fig.add_trace(go.Histogram(x=raw_words, name="raw", opacity=0.55, nbinsx=35, marker_color="#7aa6ff"), row=1, col=2)
fig.add_trace(go.Histogram(x=clean_words, name="clean", opacity=0.55, nbinsx=35, marker_color="#2d6cdf"), row=1, col=2)

fig.add_trace(go.Bar(x=label_compare.index.astype(str), y=label_compare["raw"], name="raw labels", marker_color="#7aa6ff"), row=2, col=1)
fig.add_trace(go.Bar(x=label_compare.index.astype(str), y=label_compare["clean"], name="clean labels", marker_color="#2d6cdf"), row=2, col=1)

fig.add_trace(go.Histogram(x=changed["delta"], nbinsx=35, marker_color="#1f4e79", showlegend=False), row=2, col=2)

fig.update_layout(
    template="plotly_white",
    height=900,
    barmode="overlay",
    title="Before vs after preprocessing",
)
fig.update_xaxes(title_text="issue type", row=1, col=1)
fig.update_xaxes(title_text="word count", row=1, col=2)
fig.update_xaxes(title_text="sentiment_llm", row=2, col=1)
fig.update_xaxes(title_text="raw char count - clean char count", row=2, col=2)
fig.update_yaxes(title_text="rows", row=1, col=1)
fig.update_yaxes(title_text="rows", row=1, col=2)
fig.update_yaxes(title_text="rows", row=2, col=1)
fig.update_yaxes(title_text="rows", row=2, col=2)
fig.show()

In [8]:
changed = clean_df.loc[clean_df["content_raw"].fillna("") != clean_df["content"].fillna("")].copy()
changed["raw_len"] = changed["content_raw"].fillna("").str.len()
changed["clean_len"] = changed["content"].fillna("").str.len()
changed["delta"] = changed["raw_len"] - changed["clean_len"]
changed["word_delta"] = changed["content_raw"].fillna("").str.split().map(len) - changed["content"].fillna("").str.split().map(len)

top_changed = changed.sort_values("delta", ascending=False).head(10)
display(top_changed[["review_id", "product_name", "sentiment_llm", "raw_len", "clean_len", "delta", "content_raw", "content"]])

fig = px.scatter(
    changed.sample(min(len(changed), 800), random_state=42),
    x="raw_len",
    y="clean_len",
    color="sentiment_llm",
    opacity=0.55,
    title="Raw vs clean length on changed rows",
    labels={"raw_len": "raw char count", "clean_len": "clean char count"},
)
fig.update_layout(template="plotly_white")
fig.show()

product_summary = (
    changed.groupby("product_name")
    .agg(rows=("review_id", "count"), avg_delta=("delta", "mean"), avg_words_delta=("word_delta", "mean"))
    .sort_values(["rows", "avg_delta"], ascending=[False, False])
    .head(15)
    .reset_index()
)
display(product_summary)

,review_id,product_name,sentiment_llm,raw_len,clean_len,delta,content_raw,content
1137,12593634,Sách Sinh Học Phân Tử Của Tế Bào - Sự Sinh Trưởng Và Phát Triển Của Tế Bào (Tập 5),1,255,36,219,Sách được bao gói tốt. Tớ chưa đọc...................................................................................................,Sách được bao gói tốt. Tớ chưa đọc..
1010,10645258,Combo 2 Cuốn: How To Crack The IELTS Speaking +Writing Test,2,1907,1710,197,"Nhìn chung, đây là bộ sách hay được đầu tư công phu về mặt thiết kế và nội dung. Sau 1 tuần sử dụng, mình nhận thấy có sự khác nhau khá ...","Nhìn chung, đây là bộ sách hay được đầu tư công phu về mặt thiết kế và nội dung. Sau 1 tuần sử dụng, mình nhận thấy có sự khác nhau khá ..."
2758,9948743,Sách Muôn Kiếp Nhân Sinh 2,1,3185,3042,143,"(Quan điểm cá nhân)\n- Đánh giá nội dung: Khá\n- Loại sách: xếp vào loại sách tiểu thuyết với chất liệu lịch sử, luân hồi, nhân quả..., ...","(Quan điểm cá nhân) - Đánh giá nội dungemoji_kh_lo_i_s_ch xếp vào loại sách tiểu thuyết với chất liệu lịch sử, luân hồi, nhân quả.., sẽ ..."
6402,760289,Sách Thiền Và Nghệ Thuật Bảo Dưỡng Xe Máy,2,2905,2764,141,Cuốn sách là cuộc hành trình đi tìm ý nghĩa hay định nghĩa Phẩm Chất (Quality) của nhân vật tên là Phaedrus. Phẩm chất (quality) là gì? ...,Cuốn sách là cuộc hành trình đi tìm ý nghĩa hay định nghĩa Phẩm Chất (Quality) của nhân vật tên là Phaedrus. Phẩm chất (quality) là gì? ...
9881,8244920,Sách Chuyện Nhỏ Sài Gòn Bao Nhớ,0,1076,947,129,Không biết đã có khách hàng nào góp ý về số lượng emails Tiki chuyển tải tới khách mua cho mỗi đơn hàng hay không ? tôi...,Không biết đã có khách hàng nào góp ý về số lượng emails Tiki chuyển tải tới khách mua cho mỗi đơn hàng hay không ? tôi đặt mua 1 đơn hà...
6663,5386929,Sách Hai Mặt Của Gia Đình,2,2678,2557,121,"Khi nhắc tới chủ đề gia đình, mình nghĩ ngay tới cuốn Hai mặt của gia đình – Choi Kwanghyun. Sách theo thể loại tâm lý học, thêm những c...","Khi nhắc tới chủ đề gia đình, mình nghĩ ngay tới cuốn Hai mặt của gia đình – Choi Kwanghyun. Sách theo thể loại tâm lý học, thêm những c..."
4031,8300170,Sách Án Mạng Mười Một Chữ,0,764,657,107,"Truyện nhiều nhân vật quá rối cho người đọc. Còn về nội dung thì theo kiểu penthouse quá, nó là cho người đọc cảm thấ...","Truyện nhiều nhân vật quá rối cho người đọc. Còn về nội dung thì theo kiểu penthouse quá, nó là cho người đọc cảm thấm tác giả quá ép bu..."
4324,770365,Truyện đọc tiếng Anh - Wordsworth Editions: The Little Prince,1,768,666,102,Một cuốn sách ngoại văn 50k thì rất khó để yêu cầu nó hoàn mỹ được rồi =)))\nCó vài bạn sẽ phàn nàn giấy mỏng dín...,"Một cuốn sách ngoại văn 50k thì rất khó để yêu cầu nó hoàn mỹ được rồi =))) Có vài bạn sẽ phàn nàn giấy mỏng dính, màu tối như sách cũng..."
8438,10304804,Sách Khi Lỗi Thuộc Về Những Vì Sao (Tái Bản),0,699,603,96,Theo cảm nhận của riêng mình cuốn sách này k hay và cũng k thực sự đặc sắc. Lúc đầu nghe mô tả mình đã nghĩ nó sẽ c...,Theo cảm nhận của riêng mình cuốn sách này không hay và cũng không thực sự đặc sắc. Lúc đầu nghe mô tả mình đã nghĩ nó sẽ cảm động và th...
4504,1178542,Sách Tự Học Tiếng Nhật Dành Cho Người Mới Bắt Đầu (Kèm CD Hoặc Dùng App),0,135,50,85,"TIKI giao hàng không đúng đơn hàng,không có CD.\n.......................................................................................","TIKI giao hàng không đúng đơn hàng,không có cd. .."


,product_name,rows,avg_delta,avg_words_delta
0,Mình Chỉ Là Người Bình Thường (Sách Tô Màu),131,-4.526718,0.198473
1,Sách Hai Mặt Của Gia Đình,81,-1.222222,5.740741
2,Sách Barron's Essential Words For The Toeic (Tái Bản),59,-2.271186,1.559322
3,Sách Sức Mạnh Tiềm Thức (Tái Bản) (Bìa Mềm),55,-4.563636,0.309091
4,Combo Doraemon - Truyện Dài (Trọn Bộ 24 Cuốn) - Tái Bản,54,-1.185185,0.000000
5,Sách Những Người Hàng Xóm - Bản Thường - Bìa Mềm (Tặng kèm Bookmark + Thẻ treo),44,-1.818182,3.159091
6,Sách Đắc Nhân Tâm (Khổ Lớn),44,-3.340909,0.045455
7,"\t Sách Mùa hè không tên (Nguyễn Nhật Ánh) - Tặng kèm Bookmark, Tranh màu",43,-11.046512,0.000000
8,Sách Xứ Cát,42,-7.261905,1.452381
9,Sách Tôn Tử Binh Pháp ( Tái Bản ),40,-2.450000,0.075000


## Takeaways

- `content` dược làm sạch mạnh phần kỹ thuật, nhưng vẫn giữ tín hiệu sentiment như emoji và dấu nhấn.
- Label distribution gần như giữ nguyên sau clean.
- `raw_len - clean_len` cho thấy preprocessing chủ yếu dọn noise, không làm mất quá nhiều nội dung.
- Những product có nhiều review và delta lớn thường là nơi dữ liệu bẩn hoặc có tín hiệu cảm xúc mạnh.